# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema at the following URL:

**https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json**

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset metadata from the Croissant schema
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display main metadata attributes
print(f"Dataset name: {metadata.name}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Description: {metadata.description}\n")
print(f"License: {getattr(metadata, 'license', 'N/A')}")
print(f"Citation: {getattr(metadata, 'citeAs', 'N/A')}")

## 2. Data Overview
Review all record sets and fields available within the dataset. All entities are referenced by their `@id` fields.

In [ ]:
# List available record sets by their @id and show fields/columns for each
print("\nRecord Sets found in the dataset:")
record_sets = []
for recordset in getattr(metadata, "recordSet", []):
    r_id = getattr(recordset, "@id", None)
    print(f"- RecordSet @id: {r_id}  | Name: {getattr(recordset, 'name', '')}")
    record_sets.append(r_id)
    # List fields and columns, referenced by @id
    fields = getattr(recordset, 'field', [])
    if len(fields) > 0:
        print("  Fields in this RecordSet:")
        for f in fields:
            print(f"    - {getattr(f, '@id', f)}: {getattr(f, 'name', '')}")
    columns = getattr(recordset, 'column', [])
    if len(columns) > 0:
        print("  Columns in this RecordSet:")
        for c in columns:
            print(f"    - {getattr(c, '@id', c)}: {getattr(c, 'name', '')}")
    print()
if not record_sets:
    print("No record sets defined in schema (record sets could be bundled in distributions/files).")
# If no record sets, see if the dataset has only one main file
if not record_sets:
    print("Attempting to fetch main file via dataset.distribution...")
    print("Distributions:")
    for dist in getattr(metadata, 'distribution', []):
        print(f"- Distribution @id: {getattr(dist, '@id', dist)}  | Name: {getattr(dist, 'name', '')}")

## 3. Data Extraction
Load records from a record set (or from the main file if no explicit record sets exist) into a pandas DataFrame.

Since the Croissant package for this dataset currently does not define any explicit record sets, we will extract all available records (default record set) and display their field names.

In [ ]:
# Bulk-read records using the default record set (top-level tabular dataset)
all_records = list(dataset.records())

# Load into DataFrame
df = pd.DataFrame(all_records)
print(f"Total records loaded: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply analysis and data processing steps. We will select numeric fields (`'NormalizedAbundance(StimulatedMastCell)'` and `'MW'` if available), filter, normalize, and perform groupby operations.

Columns should always be referenced by their column names found above, as each corresponds to a field or column `@id`.

In [ ]:
# Choose a numeric field present in the data
candidate_numeric_fields = ['NormalizedAbundance(StimulatedMastCell)', 'NormalizedAbundance(UnstimulatedMastCell)', 'MW', 'PeptideCount', 'pI']
numeric_field = None
for col in candidate_numeric_fields:
    if col in df.columns:
        numeric_field = col
        break

if numeric_field is None:
    raise ValueError("No recognized numeric field found in dataset for demonstration.")

print(f"Using numeric field '{numeric_field}' for EDA.\n")
# Coerce to numeric, errors='coerce' turns problematic values into NaN
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
threshold = df[numeric_field].quantile(0.8)
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered {len(filtered_df)} records with {numeric_field} > {threshold:.2f} (top 20%).")

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / (filtered_df[numeric_field].std() + 1e-8)
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Attempt groupby (e.g., by 'Description' if present)
group_field = None
possible_group_fields = ['Description', 'Accession', 'Modification']
for col in possible_group_fields:
    if col in filtered_df.columns:
        group_field = col
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped mean '{numeric_field}' by '{group_field}':")
    print(grouped_df.head())
else:
    print("No group field found; skipping groupby.")

## 5. Visualization
Visualize data distributions or relationships using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the chosen numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True, color='steelblue')
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Boxplot for normalized values if group_field is available
if group_field:
    plt.figure(figsize=(10, 5))
    top_groups = filtered_df[group_field].value_counts().index[:10]
    sns.boxplot(
        data=filtered_df[filtered_df[group_field].isin(top_groups)],
        x=group_field,
        y=numeric_field,
    )
    plt.title(f"{numeric_field} by {group_field} (Top 10)")
    plt.xticks(rotation=45, ha='right')
    plt.show()

## 6. Conclusion
- The FAIR^2 Mass Spectrometry dataset contains protein information with columns including accession, description, molecular weight, peptide counts, pI (isoelectric point), and normalized abundance metrics per sample.
- We extracted and analyzed the main tabular file; no explicit record sets were present in the Croissant schema, so we used the default dataset.
- We filtered and normalized a numerical field, and visualized its distribution, showing the rich range of protein abundances measured.
- Further domain-specific statistical or machine learning analyses can be performed now that the data is structured as a pandas DataFrame.